# 02 · Trace the circuit

 **Question:** Can we reproduce the compiled transformer's calculations by following
 its attention and MLP blocks manually?

 This notebook covers all twelve programs plus the A/B examples. It runs independently;
 [01 · Programs](01_programs.ipynb) explains their definitions and writer schedules.

## Configure
Choose the compiler settings, input samples, and trace to inspect.

In [ ]:
import sys
from pathlib import Path

# Run from the repository root or any folder inside it.
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (root for root in (start, *start.parents) if (root / "raspformer").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the RaspFormer repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from raspformer.compiler import CompilerConfig, run_trace_study
from raspformer.programs import build_programs

In [ ]:
config = CompilerConfig(
    vocab=tuple(range(-2, 6)),
    max_seq_len=6,
    causal=False,
    bos="BOS",
    pad="PAD",
    mlp_exactness=100,
    rel_tol=1e-4,
    abs_tol=1e-4,
)
samples = [
    [1, 2, 1],
    [-2, 0, 3, 1],
    [5],
    [0, 0, 0, 0],
    [5, 4, 3, 2, 1, 0],
]

OUTPUT_ROOT = PROJECT_ROOT / "results" / "trace"
FOCUS_PROGRAM = "B_histogram_then_repeated_class"
FOCUS_SAMPLE = 0
PROGRAMS = build_programs(config.max_seq_len, include_examples=True)

## Run

 Construct the manual embeddings, apply each active Craft block, and compare with
 the assembled model. Embeddings, output scores, and decoded answers are strict
 checks at the configured tolerance. Intermediate residual differences are diagnostics.

In [ ]:
result = run_trace_study(PROGRAMS, config, samples, OUTPUT_ROOT)
evidence = result["evidence"]
display(pd.DataFrame(evidence["programs"]))
print(f"Saved traces: {result['run_dir']}")

## Inspect

 Each residual snapshot includes the BOS row. Decoded outputs contain only input
 positions. The manifest records residual lane labels; `traces.json` contains the
 detailed snapshots and output scores.

In [ ]:
examples = [row for row in result["traces"] if row["program"] == FOCUS_PROGRAM]
example = examples[FOCUS_SAMPLE]
print(f"Input: {example['tokens']}")
print(f"RASP: {example['rasp_output']}; decoded model: {example['model_decoded']}")
display(
    pd.DataFrame(
        [
            {
                "block": row["module"],
                "residual_shape": np.shape(row["residual"]),
                "output_scores": row["output_scores"],
            }
            for row in example["block_trace"]
        ]
    )
)
print(f"Maximum absolute output-score error: {example['max_abs_score_error']:.3g}")

In [ ]:
warnings = pd.DataFrame(evidence["residual_warnings"])
if warnings.empty:
    print("No residual discrepancies at the configured tolerance.")
else:
    display(warnings)
print(
    f"{len(result['traces'])} strict output checks; {len(warnings)} residual diagnostics."
)

## Interpret

 Passing output checks establishes agreement on the sampled inputs. Review residual
 warnings before claiming agreement throughout the circuit. Zero warnings means the
 sampled residuals agree at the configured tolerance, not that every possible input
 has been checked.

 **Next:** [Measure representation geometry](03_geometry.ipynb).